# Kubeflow - VertexAI pipelines tutorial
## Installing required libraries

In [11]:
import ensurepip
import subprocess
import sys

try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    ensurepip.bootstrap(upgrade=True)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "google-auth>=2.30.0",
    "google-cloud-aiplatform>=1.82,<2.0",
    "kfp>=2.11,<3.0",
])


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


0

In [ ]:
import importlib.metadata
import sys

import google.cloud.aiplatform as aiplatform
import kfp

print(f"Python executable: {sys.executable}")
print(f"KFP SDK version: {kfp.__version__}")
print(f"google-cloud-aiplatform version: {importlib.metadata.version('google-cloud-aiplatform')}")

## Define your values

In [12]:
import random
import string
PROJECT_ID = "project3grupo2"
LOCATION = "us-central1"
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8)) # Comenta esto y reemplaza con el valor que se imprime al ejecutar la celda para evitar multiples buckets
print("Este es el valor a reemplazar en random_suffix: "+str(random_suffix))

BUCKET_NAME = f"{PROJECT_ID}-bucket-{random_suffix}"
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root/"

BQ_LOCATION = LOCATION.split("-")[0].upper()
BUCKET_URI = "gs://"+BUCKET_NAME

Este es el valor a reemplazar en random_suffix: 9fkbzd5y


In [13]:
# Create a bucket:
! gcloud storage buckets create gs://$BUCKET_NAME --location=$LOCATION --uniform-bucket-level-access

Creating gs://project3grupo2-bucket-9fkbzd5y/...


In [14]:
# Active gcloud account and service account, when available
shell_output = !gcloud auth list --filter=status:ACTIVE --format="value(account)" 2>/dev/null
ACTIVE_ACCOUNT = shell_output[0].strip()

if ACTIVE_ACCOUNT.endswith(".gserviceaccount.com"):
    SERVICE_ACCOUNT = ACTIVE_ACCOUNT
    STORAGE_IAM_MEMBER = f"serviceAccount:{ACTIVE_ACCOUNT}"
else:
    SERVICE_ACCOUNT = None
    STORAGE_IAM_MEMBER = f"user:{ACTIVE_ACCOUNT}"

print(f"Active account: {ACTIVE_ACCOUNT}")
print(f"Storage IAM member: {STORAGE_IAM_MEMBER}")
print(f"Vertex service account: {SERVICE_ACCOUNT}")

Active account: paolareguer@gmail.com
Storage IAM member: user:paolareguer@gmail.com
Vertex service account: None


In [15]:
! gcloud storage buckets add-iam-policy-binding gs://$BUCKET_NAME \
    --member="$STORAGE_IAM_MEMBER" \
    --role="roles/storage.objectAdmin"

bindings:
- members:
  - projectEditor:project3grupo2
  - projectOwner:project3grupo2
  role: roles/storage.legacyBucketOwner
- members:
  - projectViewer:project3grupo2
  role: roles/storage.legacyBucketReader
- members:
  - projectEditor:project3grupo2
  - projectOwner:project3grupo2
  role: roles/storage.legacyObjectOwner
- members:
  - projectViewer:project3grupo2
  role: roles/storage.legacyObjectReader
- members:
  - user:paolareguer@gmail.com
  role: roles/storage.objectAdmin
etag: CAI=
kind: storage#policy
resourceId: projects/_/buckets/project3grupo2-bucket-9fkbzd5y
version: 1


## Initialize Vertex AI pipelines

In [16]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component

In [18]:
# Authenticate Application Default Credentials for the Vertex AI Python SDK
# If this opens a browser login, complete it with the same Google account.
! gcloud auth application-default login
! gcloud auth application-default set-quota-project $PROJECT_ID

import google.auth

credentials, adc_project = google.auth.default()
print(f"ADC project: {adc_project}")
print(f"Quota project: {getattr(credentials, 'quota_project_id', None)}")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=WBSqtHEWV7qCSgqWYA0phijDEZQHzC&access_type=offline&code_challenge=ea9rOdExneHKMrOqb6dNSy4QU6YDoiuyp2I_paHDeNw&code_challenge_method=S256


Credentials saved to file: [/Users/paolareguera/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "project3grupo2" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.

Credentials saved to file: [/Users/paolaregu

In [19]:
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

# Exercise: build a training pipeline 

In [20]:
@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def ingest_data(X_out: Output[Dataset], y_out: Output[Dataset]):
    # Ejemplo: guardar cada artifact como un fichero usando el patron recomendado en KFP 2.x.
    from sklearn.datasets import load_iris
    import numpy as np

    X_out.path += ".npy"
    y_out.path += ".npy"

    iris = load_iris()
    np.save(X_out.path, iris.data)
    np.save(y_out.path, iris.target)

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def split_data(
    X_in: Input[Dataset],
    y_in: Input[Dataset],
    X_train_out: Output[Dataset],
    X_test_out: Output[Dataset],
    y_train_out: Output[Dataset],
    y_test_out: Output[Dataset],
    test_size: float = 0.2
):
    from sklearn.model_selection import train_test_split
    import numpy as np

    X = np.load(X_in.path)
    y = np.load(y_in.path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=0)

    X_train_out.path += ".npy"
    X_test_out.path += ".npy"
    y_train_out.path += ".npy"
    y_test_out.path += ".npy"

    np.save(X_train_out.path, X_train)
    np.save(X_test_out.path, X_test)
    np.save(y_train_out.path, y_train)
    np.save(y_test_out.path, y_test)

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def train(
    X_train: Input[Dataset],
    y_train: Input[Dataset],
    model_out: Output[Model],
    max_depth: int,
    n_estimators: int,
    random_state: int,
):
    from sklearn.ensemble import RandomForestClassifier
    import pickle, numpy as np

    X = np.load(X_train.path)
    y = np.load(y_train.path)

    clf = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators, random_state=random_state)
    clf.fit(X, y)

    model_out.path += ".pkl"
    with open(model_out.path, "wb") as f:
        pickle.dump(clf, f)

@component(base_image="python:3.11-slim", packages_to_install=['scikit-learn==1.5.2', 'numpy==1.26.4'])
def show_metrics(
    model: Input[Model],
    X_test: Input[Dataset],
    y_test: Input[Dataset],
    metrics: Output[Metrics],
):
    from sklearn.metrics import accuracy_score, f1_score
    import pickle, numpy as np

    X = np.load(X_test.path)
    y = np.load(y_test.path)

    with open(model.path, "rb") as f:
        clf = pickle.load(f)

    y_pred = clf.predict(X)
    metrics.log_metric("accuracy", accuracy_score(y, y_pred))
    metrics.log_metric("f1_macro", f1_score(y, y_pred, average="macro"))

@dsl.pipeline
def training_pipeline(max_depth: int = 2, n_estimators: int = 100, random_state: int = 0):
    ingest = ingest_data()

    split = split_data(
        X_in=ingest.outputs['X_out'],
        y_in=ingest.outputs['y_out'],
    )

    model = train(
        X_train=split.outputs['X_train_out'],
        y_train=split.outputs['y_train_out'],
        max_depth=max_depth,
        n_estimators=n_estimators,
        random_state=random_state,
    )

    show_metrics(
        model=model.outputs['model_out'],
        X_test=split.outputs['X_test_out'],
        y_test=split.outputs['y_test_out'],
    )

compiler.Compiler().compile(pipeline_func=training_pipeline, package_path='training_pipeline.yaml')


In [21]:
job = aiplatform.PipelineJob(
    display_name="training_pipeline",
    template_path="training_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    project=PROJECT_ID,
    location=LOCATION,
    enable_caching=True,
)

job.run(service_account=SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/training-pipeline-20260521160325?project=29441691460
PipelineJob projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325 current state:
3
PipelineJob projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325 current state:
3
PipelineJob projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325 current state:
3
PipelineJob projects/29441691460/locations/us-central1/pipelineJobs/training-pipeline-20260521160325 current state:
3
PipelineJob projects/29441691460/locations/us-centra